# Export RandomForest Model for Live Trading

Loads `data/latest_features.jsonl`, trains RandomForest with optimal features and hyperparameters
from `data/optimal_features_rf.json`, exports to `models/`.


In [1]:
import sys

sys.path.insert(0, str(__import__("pathlib").Path.cwd().parent))

import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

random.seed(42)
np.random.seed(42)

FEATURES_PATH = Path("../../data/latest_features.jsonl")
CONFIG_PATH = Path("../../data/optimal_features_rf.json")
MODELS_DIR = Path("../../models")

## 1. Load data and optimal config

In [2]:
rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

with open(CONFIG_PATH) as f:
    config = json.load(f)
    feat_cols = config["features"]
    hyperparams = config.get("hyperparameters", {})

df[feat_cols] = df[feat_cols].fillna(0.0)

print(f"Loaded {len(df):,} rows, {df['candle_id'].nunique()} candles")
print(f"Features: {len(feat_cols)} (from {CONFIG_PATH.name})")
print(f"Hyperparameters: {hyperparams}")
print(f"UP rate: {df['target'].mean() * 100:.1f}%")

Loaded 66,153 rows, 1376 candles
Features: 15 (from optimal_features_rf.json)
Hyperparameters: {'n_estimators': 200, 'max_depth': 15, 'min_samples_leaf': 20, 'random_state': 42}
UP rate: 51.1%


## 2. Train/val split — validation metrics

In [3]:
candle_ids = df["candle_id"].unique()
split_idx = int(len(candle_ids) * 0.8)
train_ids = set(candle_ids[:split_idx])

df_train = df[df["candle_id"].isin(train_ids)]
df_val = df[~df["candle_id"].isin(train_ids)]

scaler = StandardScaler()
X_train = scaler.fit_transform(df_train[feat_cols].values)
y_train = df_train["target"].values
X_val = scaler.transform(df_val[feat_cols].values)
y_val = df_val["target"].values

model = RandomForestClassifier(n_jobs=-1, **hyperparams)
model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)
val_acc = accuracy_score(y_val, val_preds)

print(f"Validation ({df_val['candle_id'].nunique()} candles, {len(df_val):,} rows):")
print(f"  Snapshot accuracy: {val_acc * 100:.1f}%")
print()
print(classification_report(y_val, val_preds, target_names=["DOWN", "UP"]))

Validation (276 candles, 12,917 rows):
  Snapshot accuracy: 75.3%

              precision    recall  f1-score   support

        DOWN       0.78      0.70      0.74      6409
          UP       0.73      0.80      0.77      6508

    accuracy                           0.75     12917
   macro avg       0.76      0.75      0.75     12917
weighted avg       0.76      0.75      0.75     12917



## 3. Train on ALL data and export

In [4]:
scaler = StandardScaler()
X_all = scaler.fit_transform(df[feat_cols].values)
y_all = df["target"].values

model = RandomForestClassifier(n_jobs=-1, **hyperparams)
model.fit(X_all, y_all)

model_path = MODELS_DIR / "rf_v1.joblib"
scaler_path = MODELS_DIR / "rf_scaler_v1.joblib"
feat_path = MODELS_DIR / "rf_feature_cols_v1.joblib"

joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)
joblib.dump(feat_cols, feat_path)

print(f"Saved to {MODELS_DIR}/:")
for p in [model_path, scaler_path, feat_path]:
    print(f"  {p.name}: {p.stat().st_size:,} bytes")

Saved to ../../models/:
  rf_v1.joblib: 28,704,969 bytes
  rf_scaler_v1.joblib: 927 bytes
  rf_feature_cols_v1.joblib: 298 bytes


## 4. Verify exported model

In [5]:
loaded_model = joblib.load(MODELS_DIR / "rf_v1.joblib")
loaded_scaler = joblib.load(MODELS_DIR / "rf_scaler_v1.joblib")
loaded_features = joblib.load(MODELS_DIR / "rf_feature_cols_v1.joblib")

sample = df[feat_cols].iloc[:10].values
X_sample = loaded_scaler.transform(sample)
probs = loaded_model.predict_proba(X_sample)[:, 1]

print(f"Features: {len(loaded_features)}")
print("Sample predictions:")
for i in range(10):
    print(f"  Row {i}: prob={probs[i]:.3f}  truth={df['target'].iloc[i]}")

print("\nModel ready for live trading.")

Features: 15
Sample predictions:
  Row 0: prob=0.560  truth=1
  Row 1: prob=0.485  truth=1
  Row 2: prob=0.459  truth=1
  Row 3: prob=0.449  truth=1
  Row 4: prob=0.610  truth=1
  Row 5: prob=0.618  truth=1
  Row 6: prob=0.446  truth=1
  Row 7: prob=0.518  truth=1
  Row 8: prob=0.585  truth=1
  Row 9: prob=0.611  truth=1

Model ready for live trading.


## Conclusion

Exported RandomForest model to `models/`.
Run `rf/03_strategy.ipynb` for strategy discovery on forward-test candles.
